# 05 - Generative AI with Azure OpenAI

**AI-900 Domain:** Describe features of generative AI workloads on Azure

## What You'll Learn
- How **GPT models** work (chat completions)
- The **system/user/assistant** message roles
- **Prompt Engineering** techniques:
  - Zero-shot prompting
  - Few-shot prompting
  - Chain-of-thought reasoning
- How Azure OpenAI keeps AI safe with **content filters**

> **Just click `Run All` - no coding needed!**

## Setup: Load Credentials

In [ ]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv(os.path.join("..", ".env"))

OPENAI_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
OPENAI_KEY = os.environ.get("AZURE_OPENAI_KEY", "")
DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini")

if not OPENAI_KEY:
    raise ValueError("AZURE_OPENAI_KEY is not set! Please check your .env file.")

# Create the Azure OpenAI client
client = AzureOpenAI(
    azure_endpoint=OPENAI_ENDPOINT,
    api_key=OPENAI_KEY,
    api_version="2024-06-01"
)

print(f"Azure OpenAI client created!")
print(f"Deployment: {DEPLOYMENT}")

---
## Part 1: Understanding Chat Completions

GPT models use a **chat format** with three types of messages:

| Role | Purpose | Example |
|------|---------|--------|
| **system** | Sets the AI's behavior and personality | "You are a helpful assistant" |
| **user** | The question or instruction from the human | "What is machine learning?" |
| **assistant** | The AI's response (or a previous response for context) | "Machine learning is..." |

Let's make our first chat request!

In [ ]:
def chat(messages, temperature=0.7, max_tokens=500):
    """Send a chat request to Azure OpenAI and display the response."""
    response = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )
    reply = response.choices[0].message.content
    tokens_used = response.usage.total_tokens
    print(f"AI Response ({tokens_used} tokens used):")
    print("-" * 50)
    print(reply)
    print("-" * 50)
    return reply

# Simple chat
print("Question: What is artificial intelligence? Explain in 2-3 sentences.\n")
_ = chat([
    {"role": "system", "content": "You are a helpful AI tutor for students studying for the AI-900 exam."},
    {"role": "user", "content": "What is artificial intelligence? Explain in 2-3 sentences."}
])

### System Messages: Changing the AI's Personality

The **system message** dramatically changes how the AI responds. Let's ask the same question with different system prompts.

In [ ]:
question = "What is machine learning?"

system_prompts = [
    ("Explains like a teacher", "You are a patient teacher explaining to a 10-year-old. Use simple words and fun examples."),
    ("Explains like a scientist", "You are a computer science professor. Use precise technical language."),
    ("Explains as a poet", "You are a poet. Explain concepts using metaphors and poetic language. Keep it brief.")
]

for title, system_prompt in system_prompts:
    print(f"\n{'='*50}")
    print(f"System: {title}")
    print(f"{'='*50}")
    _ = chat([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ], max_tokens=150)

---
## Part 2: Prompt Engineering Techniques

**Prompt Engineering** is the art of writing effective prompts to get better results from AI models. This is a key AI-900 topic!

### Technique 1: Zero-Shot Prompting
Ask the model to do something with **no examples** - just instructions.

In [ ]:
print("=" * 50)
print("ZERO-SHOT: No examples, just instructions")
print("=" * 50)

_ = chat([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Classify the following text as POSITIVE, NEGATIVE, or NEUTRAL:\n\nText: 'The new update made the app much faster and easier to use!'\n\nClassification:"}
])

### Technique 2: Few-Shot Prompting
Give the model a **few examples** so it understands the pattern you want.

In [ ]:
print("=" * 50)
print("FEW-SHOT: Providing examples to guide the model")
print("=" * 50)

_ = chat([
    {"role": "system", "content": "You classify customer feedback into categories."},
    {"role": "user", "content": "Feedback: 'Delivery was late by 3 days'"},
    {"role": "assistant", "content": "Category: SHIPPING | Sentiment: NEGATIVE"},
    {"role": "user", "content": "Feedback: 'Great product quality, love the color'"},
    {"role": "assistant", "content": "Category: PRODUCT | Sentiment: POSITIVE"},
    {"role": "user", "content": "Feedback: 'Customer support resolved my issue quickly'"},
    {"role": "assistant", "content": "Category: SUPPORT | Sentiment: POSITIVE"},
    {"role": "user", "content": "Feedback: 'The website keeps crashing when I try to checkout'"}
])

### Technique 3: Chain-of-Thought (CoT)
Ask the model to **think step by step**, which improves accuracy for complex reasoning.

In [ ]:
print("=" * 50)
print("WITHOUT Chain-of-Thought")
print("=" * 50)

_ = chat([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "A store has 5 apples. They buy 3 bags of apples with 8 apples each. A customer buys 12 apples. Then another shipment of 15 apples arrives. How many apples does the store have?"}
], max_tokens=100)

print("\n")
print("=" * 50)
print("WITH Chain-of-Thought")
print("=" * 50)

_ = chat([
    {"role": "system", "content": "You are a helpful assistant. Think step by step and show your work."},
    {"role": "user", "content": "A store has 5 apples. They buy 3 bags of apples with 8 apples each. A customer buys 12 apples. Then another shipment of 15 apples arrives. How many apples does the store have? Let's think step by step."}
], max_tokens=300)

---
## Part 3: Temperature - Controlling Creativity

**Temperature** controls how creative or focused the AI's responses are:
- **0.0** = Very focused, deterministic (same answer every time)
- **0.7** = Balanced (default)
- **1.0** = Very creative, varied responses

Let's compare!

In [ ]:
question = "Write a one-sentence description of cloud computing."

for temp in [0.0, 0.5, 1.0]:
    print(f"\n{'='*50}")
    print(f"Temperature: {temp}")
    print(f"{'='*50}")
    _ = chat([
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": question}
    ], temperature=temp, max_tokens=80)

---
## Part 4: Multi-Turn Conversation

GPT models can maintain context across multiple turns of conversation by including the full message history.

In [ ]:
print("=" * 50)
print("MULTI-TURN CONVERSATION")
print("=" * 50)

conversation = [
    {"role": "system", "content": "You are an AI tutor helping students study for the AI-900 exam. Be concise."}
]

# Turn 1
conversation.append({"role": "user", "content": "What are the main types of machine learning?"})
print("\nStudent: What are the main types of machine learning?")
response1 = chat(conversation, max_tokens=200)
conversation.append({"role": "assistant", "content": response1})

# Turn 2 - References previous answer
conversation.append({"role": "user", "content": "Can you give me a real-world example of the second type you mentioned?"})
print("\nStudent: Can you give me a real-world example of the second type you mentioned?")
response2 = chat(conversation, max_tokens=200)
conversation.append({"role": "assistant", "content": response2})

# Turn 3 - Continues context
conversation.append({"role": "user", "content": "Would that example appear on the AI-900 exam?"})
print("\nStudent: Would that example appear on the AI-900 exam?")
response3 = chat(conversation, max_tokens=200)

---
## Key Concepts for AI-900

| Concept | Description |
|---------|-------------|
| **GPT** | Generative Pre-trained Transformer - a type of large language model |
| **Chat Completions** | The API format for interacting with GPT models |
| **System Message** | Sets the AI's behavior, personality, and constraints |
| **Temperature** | Controls randomness: 0 = focused, 1 = creative |
| **Tokens** | Units of text (roughly 3/4 of a word) - impacts cost and limits |
| **Zero-Shot** | Asking the model to perform a task with no examples |
| **Few-Shot** | Providing examples to guide the model's output format |
| **Chain-of-Thought** | Asking the model to reason step-by-step |
| **Content Filters** | Azure OpenAI's built-in safety system that blocks harmful content |

### AI-900 Exam Tips
- **Azure OpenAI Service** provides access to GPT, DALL-E, and Whisper models
- It's a **managed service** - Microsoft handles the infrastructure
- **Prompt engineering** is the skill of writing effective prompts (the exam tests this!)
- Azure OpenAI includes **content filtering** by default (Responsible AI in practice)
- **Tokens** are the billing unit - more tokens = higher cost
- The model does NOT "remember" previous conversations unless you include the history

---
**Next:** Open `06_document_intelligence.ipynb` to explore document parsing!